# C.O.A.S.T. — Train a Drowning-Detection Model (Google Colab)

**Team: The Buoyz — CANTO 2026**

This notebook fine-tunes a YOLO model to recognize swimmers / drowning at the **water surface**, using a free GPU on Google Colab. When it finishes you get a file called `best.pt` that you drop into the project (`config.py` → `MODEL_NAME = "best.pt"`).

---

## How to use this notebook (read first!)

1. Go to [colab.research.google.com](https://colab.research.google.com) → **File → Upload notebook** → upload this `.ipynb`.
2. **Turn on the free GPU:** menu **Runtime → Change runtime type → Hardware accelerator → T4 GPU → Save.** (Training on CPU is far too slow — this step is essential.)
3. Run each cell in order with the ▶ button (or `Shift+Enter`). Read the notes above each one.
4. Total time: roughly 30–90 minutes, mostly waiting on the training cell.

> **Physics reminder:** this trains for **surface** detection (heads/shoulders/struggling at the waterline). It will **not** see fully-submerged swimmers from an above-water camera — that's handled instead by your "person disappeared from the surface" distress rule. Don't expect underwater x-ray vision here.

## Step 1 — Confirm the GPU is on

This should print a table with a GPU (e.g. "Tesla T4"). If it says "command not found" or shows no GPU, go back and do the **Runtime → Change runtime type → T4 GPU** step above, then re-run.

In [ ]:
!nvidia-smi

## Step 2 — Install the tools

`ultralytics` = YOLO training. `roboflow` = downloads the labeled dataset. Takes a minute or two.

In [ ]:
!pip install -q ultralytics roboflow

import ultralytics
ultralytics.checks()  # prints versions and confirms the GPU is visible to YOLO

## Step 3 — Download the dataset from Roboflow

You need a (free) Roboflow account and the download snippet for the dataset you picked.

**How to get the snippet:**
1. Sign up at [roboflow.com](https://roboflow.com) (free).
2. Open a dataset, e.g. the big **[Swimming and Drowning Detection](https://universe.roboflow.com/drowning-detection-mgyjb/swimming-and-drowning-detection-yprco/dataset/1)** one.
3. Click **Download Dataset** → Format: **YOLOv11** → select **"Show download code"**.
4. Roboflow gives you a snippet with *your* API key already filled in. **Copy it and paste it over the placeholder below.**

It will look like this (yours will have real values):

In [ ]:
# ===== PASTE YOUR ROBOFLOW SNIPPET OVER THESE 5 LINES =====
from roboflow import Roboflow
rf = Roboflow(api_key="YOUR_API_KEY_HERE")
project = rf.workspace("WORKSPACE_NAME").project("PROJECT_NAME")
version = project.version(1)
dataset = version.download("yolov11")
# ==========================================================

# 'dataset.location' is the folder that holds data.yaml + images/ + labels/
print("Dataset downloaded to:", dataset.location)

## Step 4 — Train the model

This is the long cell (watch the GPU do its thing). A few knobs you can tune:
- `epochs` — how many times it studies the whole dataset. More = better, up to a point. 50 is a solid start; bump to 100 if you have time.
- `imgsz` — 640 matches these datasets and trains fast. 
- `batch` — how many images at once; lower this to 8 if you hit an "out of memory" error.

Starting from `yolo11s.pt` (pretrained) means we're *fine-tuning*, which is much faster than training from zero.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11s.pt")  # start from a pretrained model (fine-tuning)

results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,        # lower to 8 if you get a CUDA out-of-memory error
    patience=15,     # stop early if it stops improving
    project="coast_training",
    name="run1",
)

## Step 5 — See how well it did

This shows training graphs and the confusion matrix. The key number to look for in the training output above is **mAP50** — higher is better (0.5+ is usable, 0.8+ is strong).

In [ ]:
from IPython.display import Image, display
import os

run_dir = "coast_training/run1"
for img in ["results.png", "confusion_matrix.png", "val_batch0_pred.jpg"]:
    path = os.path.join(run_dir, img)
    if os.path.exists(path):
        print(img)
        display(Image(filename=path, width=700))

## Step 6 — Test it on YOUR OWN video

The real test: does it work on the footage *you'll* demo? Run this cell, pick a short clip from your computer when prompted, and it'll run your freshly trained model on it.

In [ ]:
from google.colab import files

print("Choose a short test video from your computer...")
uploaded = files.upload()
video_name = list(uploaded.keys())[0]

best_model = YOLO("coast_training/run1/weights/best.pt")
best_model.predict(source=video_name, save=True, conf=0.25, project="coast_preds", name="test")
print("\nDone. Annotated video saved under coast_preds/test/ — download it from the file browser on the left.")

## Step 7 — Download your trained model (`best.pt`)

This downloads `best.pt` to your computer. Then, in your project:
1. Put `best.pt` in the project folder (e.g. next to `requirements.txt`).
2. In `src/config.py`, set `MODEL_NAME = "best.pt"`.
3. Run `python src/detect.py --source data/videos/yourclip.mp4` as usual — now it uses YOUR model.

> Note: `.pt` files are git-ignored on purpose (too big for GitHub). Share `best.pt` with teammates via Google Drive, not git.

In [ ]:
from google.colab import files
files.download("coast_training/run1/weights/best.pt")